# LLM implementation

In [2]:
import torch
import torch.nn as nn
import tiktoken
import numpy as np

In [3]:

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [4]:
class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi)) *
                (x + 0.044715 * torch.pow(x, 3))
            )
        )

In [5]:

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [37]:

class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False,
    ):
        super().__init__()

        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x,attention_mask=None,past_k=None,past_v=None,use_cache=False):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(
            b, num_tokens, self.num_heads, self.head_dim
        )
        values = values.view(
            b, num_tokens, self.num_heads, self.head_dim
        )
        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim
        )

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        # (batch, n_heads, seq_len,seq_len)
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        if attention_mask is not None:
            # shape - (batch,seq_len)
            # expand to (batch,1,1,seq_len)
            padding_mask=(attention_mask.unsqueeze(1).unsqueeze(2))
            attn_scores=attn_scores.masked_fill(
                padding_mask==0,
                -torch.inf
            )
            
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )

        context_vec = self.out_proj(context_vec)

        return context_vec


In [29]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )

        self.ff = FeedForward(cfg)

        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x,attention_mask=None):

        shortcut = x
        x = self.norm1(x)
        x = self.att(x,attention_mask=attention_mask)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

In [30]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"]
        )

        self.pos_emb = nn.Embedding(
            cfg["context_length"],
            cfg["emb_dim"]
        )

        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg)
              for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False
        )

    def forward(self, in_idx,attention_mask=None):
        batch_size, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        x = tok_embeds + pos_embeds

        x = self.drop_emb(x)

        for block in self.trf_blocks:
            x=block(x,attention_mask=attention_mask)
        x = self.final_norm(x)

        logits = self.out_head(x)

        return logits

# GPT2 config

In [9]:
GPT2_SMALL_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

GPT2_MEDIUM_355M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1024,
    "n_heads": 16,
    "n_layers": 24,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

GPT2_LARGE_774M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1280,
    "n_heads": 20,
    "n_layers": 36,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

GPT2_XL_1558M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1600,
    "n_heads": 25,
    "n_layers": 48,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

In [10]:
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


CHOOSE_MODEL = "gpt2-small (124M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [11]:
file_name = "gpt2-small-124M.pth"

In [11]:


import os
import requests

url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"Downloaded to {file_name}")

Downloaded to gpt2-small-124M.pth


# load weights

In [31]:
gpt = GPTModel(BASE_CONFIG)
gpt.load_state_dict(torch.load(file_name, weights_only=True))
gpt.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device);

In [32]:
gpt

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [14]:
import tiktoken

torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")


In [15]:
import torch

PAD_TOKEN_ID=0

def texts_to_token_ids(texts, tokenizer):

    encoded = [
        tokenizer.encode(text)
        for text in texts
    ]
    max_len = max(len(seq) for seq in encoded)

    padded=[]
    attention_masks=[]

    for seq in encoded:
        pad_len=max_len-len(seq)
        padded_seq=[PAD_TOKEN_ID]*pad_len+seq
        mask=[0]*pad_len+[1]*len(seq)
        padded.append(padded_seq)
        attention_masks.append(mask)
    return (
        torch.tensor(padded),
        torch.tensor(attention_masks)
    )


def token_ids_to_texts(token_ids, tokenizer):

    return [
        tokenizer.decode(seq.tolist())
        for seq in token_ids
    ]

# sampling methods

## greedy

In [43]:

def generate(model, idx,attention_mask,max_new_tokens, context_size, eos_id=None):
    batch_size=idx.shape[0]
    finished=torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=idx.device
    )
    for _ in range(max_new_tokens):
        active_mask=~finished
        active_indices=torch.nonzero(active_mask,as_tuple=False).squeeze(-1)
        if len(active_indices)==0:
            break
        active_idx=idx[active_indices]
        active_attention_mask=attention_mask[active_indices]
        idx_cond=active_idx[:,-context_size:]
        attn_cond=active_attention_mask[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond,attention_mask=attn_cond)
        # only select the last token - newly generated token
        logits=logits[:,-1,:]
        idx_next=torch.argmax(logits,dim=-1,keepdim=True)
        new_tokens=torch.full(
            (batch_size,1),
            eos_id if eos_id is not None else 0,
            dtype=idx.dtype,
            device=idx.device
        )
        new_tokens[active_indices]=idx_next
        idx=torch.cat((idx,new_tokens),dim=1)

        new_attention=torch.zeros(
            (batch_size,1),
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        new_attention[active_indices]=1
        attention_mask=torch.cat((attention_mask,new_attention),dim=1)
        if eos_id is not None:
            newly_finished = (
                idx_next.squeeze(-1) == eos_id
            )
            finished[active_indices]|=newly_finished
        if finished.all():
            break
    return idx

In [44]:
tokenizer.eot_token

50256

In [46]:
input_ids, attention_mask = texts_to_token_ids(
    [
        "Every effort moves",
        "practice makes man"
    ],
    tokenizer
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

token_ids = generate(
    model=gpt,
    idx=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    eos_id=tokenizer.eot_token
)

In [47]:
print("generated output: \n")
for i,out in enumerate(token_ids_to_texts(token_ids,tokenizer)):
    print(f"""{i+1}. \n{out}\n""")

generated output: 

1. 
Every effort moves forward, but it's not enough.

"I'm not going to sit here and say, 'I'm not going to do this,'

2. 
practice makes man's life easier.

The first step is to understand the concept of "self-awareness."

Self-awareness is the ability to recognize



## temperature scaling

- changes the shape of the probability distribution before we select the token
- introduces a parameter T in our softmax-equation
- if T=1.0 -> its the default state - probabilities remain unchanged
- T<1.0 -> makes distribution sharper - amplifies the differences between the logits
    - rich gets ricker, and top choice dominates
    - approaches greedy as T nears 0
- T>1.0 - flattens the distribution, the math minimizes the differences boosting the probabilites of obscure , low-ranked tokens
- High temperatures (e.g., $T > 1.2$) will often cause the model to output complete gibberish by assigning too much probability to nonsensical tokens.

In [49]:

def generate(model, idx, attention_mask,max_new_tokens, context_size,temperature=0.0, eos_id=None):
    batch_size=idx.shape[0]
    finished=torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=idx.device
    )
    for _ in range(max_new_tokens):
        active_mask=~finished
        active_indices=torch.nonzero(active_mask,as_tuple=False).squeeze(-1)
        if len(active_indices)==0:
            break
        active_idx=idx[active_indices]
        active_attention_mask=attention_mask[active_indices]
        idx_cond=active_idx[:,-context_size:]
        attn_cond=active_attention_mask[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond,attention_mask=attn_cond)
        # only select the last token - newly generated token
        logits=logits[:,-1,:]
        if temperature>0.0:
            logits=logits/temperature
            logits=logits-logits.max(dim=-1,keepdim=True).values
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
        else:
            # it temp is 0. its same as greedy 
            idx_next=torch.argmax(logits,dim=-1,keepdim=True)
        new_tokens=torch.full(
            (batch_size,1),
            eos_id if eos_id is not None else 0,
            dtype=idx.dtype,
            device=idx.device
        )
        new_tokens[active_indices]=idx_next
        idx=torch.cat((idx,new_tokens),dim=1)

        new_attention=torch.zeros(
            (batch_size,1),
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        new_attention[active_indices]=1
        attention_mask=torch.cat((attention_mask,new_attention),dim=1)
        if eos_id is not None:
            newly_finished = (
                idx_next.squeeze(-1) == eos_id
            )
            finished[active_indices]|=newly_finished
        if finished.all():
            break
    return idx

In [50]:
input_ids, attention_mask = texts_to_token_ids(
    [
        "Every effort moves",
        "practice makes man"
    ],
    tokenizer
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

token_ids = generate(
    model=gpt,
    idx=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    temperature=0.5,
    eos_id=tokenizer.eot_token
)

In [51]:
print("generated output: \n")
for i,out in enumerate(token_ids_to_texts(token_ids,tokenizer)):
    print(f"""{i+1}. \n{out}\n""")

generated output: 

1. 
Every effort moves forward, but a little time needs to be made to get it done.

The project is currently in its early stages, and has been a

2. 
practice makes man-made disasters much more likely.

But if you want to know how to prevent a disaster, you need to know how to do it right



## avoid computation for finished sequences (not fully optimized tho)

In [40]:

def generate(model, idx, attention_mask,max_new_tokens, context_size,temperature=0.0, eos_id=None):
    batch_size=idx.shape[0]
    finished=torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=idx.device
    )
    for _ in range(max_new_tokens):
        active_mask=~finished
        active_indices=torch.nonzero(active_mask,as_tuple=False).squeeze(-1)
        if len(active_indices)==0:
            break
        active_idx=idx[active_indices]
        active_attention_mask=attention_mask[active_indices]
        idx_cond=active_idx[:,-context_size:]
        attn_cond=active_attention_mask[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond,attention_mask=attn_cond)
        # only select the last token - newly generated token
        logits=logits[:,-1,:]
        if temperature>0.0:
            logits=logits/temperature
            logits=logits-logits.max(dim=-1,keepdim=True).values
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
        else:
            # it temp is 0. its same as greedy 
            idx_next=torch.argmax(logits,dim=-1,keepdim=True)
        new_tokens=torch.full(
            (batch_size,1),
            eos_id if eos_id is not None else 0,
            dtype=idx.dtype,
            device=idx.device
        )
        new_tokens[active_indices]=idx_next
        idx=torch.cat((idx,new_tokens),dim=1)

        new_attention=torch.zeros(
            (batch_size,1),
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        new_attention[active_indices]=1
        attention_mask=torch.cat((attention_mask,new_attention),dim=1)
        if eos_id is not None:
            newly_finished = (
                idx_next.squeeze(-1) == eos_id
            )
            finished[active_indices]|=newly_finished
        if finished.all():
            break
    return idx

In [43]:
input_ids, attention_mask = texts_to_token_ids(
    [
        "Every effort moves",
        "practice makes man"
    ],
    tokenizer
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

token_ids = generate(
    model=gpt,
    idx=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    temperature=0.8,
    eos_id=tokenizer.eot_token
)

In [44]:
print("generated output: \n")
for i,out in enumerate(token_ids_to_texts(token_ids,tokenizer)):
    print(f"""{i+1}. \n{out}\n""")

generated output: 

1. 
Every effort moves forward. The cab drivers take their work and run with it and drive it home. It's an hour-long drive and there's nothing to get

2. 
practice makes man well acquainted with the physical and emotional differences that can make men handicapped.

(On weight loss, a woman gaining weight may feel better when



## top-k sampling

- set a number k , the model sorts the output by probability, throws away everything from rank
51 downwards and redistributed the remaining probability mass among the top 50 tokens
- it then samples from that truncated list
- controlled randomness
- eliminated the long tail of garbge tokens (can be caused by temperature), prevents model from
going off the trails
- low k might prematurely cut off perfectly valid words
- top_p only effects sampling , it doesnot effect greedy decoding much
- for top_p to take effect you must use temperature>0

In [56]:

def generate(model, idx, attention_mask,max_new_tokens, context_size,top_k=None,temperature=0.0, eos_id=None):
    batch_size=idx.shape[0]
    finished=torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=idx.device
    )
    for _ in range(max_new_tokens):
        active_mask=~finished
        active_indices=torch.nonzero(active_mask,as_tuple=False).squeeze(-1)
        if active_indices.numel()==0:
            break
        active_idx=idx[active_indices]
        active_attention_mask=attention_mask[active_indices]
        idx_cond=active_idx[:,-context_size:]
        attn_cond=active_attention_mask[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond,attention_mask=attn_cond)
        # only select the last token - newly generated token
        logits=logits[:,-1,:]
        if temperature>0.0:
            logits=logits/temperature
        if top_k is not None:
            top_logits,_=torch.topk(logits,top_k)
            min_val=top_logits[:,-1].unsqueeze(-1)
            logits=torch.where(logits<min_val,torch.tensor(float("-inf")).to(logits.device),logits)
        if temperature>0.0:
            logits=logits-logits.max(dim=-1,keepdim=True).values
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
        else:
            # it temp is 0. its same as greedy 
            idx_next=torch.argmax(logits,dim=-1,keepdim=True)
        new_tokens=torch.full(
            (batch_size,1),
            eos_id if eos_id is not None else 0,
            dtype=idx.dtype,
            device=idx.device
        )
        new_tokens[active_indices]=idx_next
        idx=torch.cat((idx,new_tokens),dim=1)

        new_attention=torch.zeros(
            (batch_size,1),
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        new_attention[active_indices]=1
        attention_mask=torch.cat((attention_mask,new_attention),dim=1)
        if eos_id is not None:
            newly_finished = (
                idx_next.squeeze(-1) == eos_id
            )
            finished[active_indices]|=newly_finished
        if finished.all():
            break
    return idx

In [59]:
input_ids, attention_mask = texts_to_token_ids(
    [
        "Every effort moves",
        "practice makes man"
    ],
    tokenizer
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

token_ids = generate(
    model=gpt,
    idx=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    top_k=40,
    temperature=0.5,
    eos_id=tokenizer.eot_token
)

In [60]:
print("generated output: \n")
for i,out in enumerate(token_ids_to_texts(token_ids,tokenizer)):
    print(f"""{i+1}. \n{out}\n""")

generated output: 

1. 
Every effort moves forward. I'm hopeful that we will be able to do this," said one of the people.

The group plans to hold a meeting at

2. 
practice makes man a better leader.

The problem is, there is no such thing as a perfect leader.

You have to start somewhere. You have



## top-p nucleus sampling

- removes the rigidity of top-k by using dynamic cutoff based on probability mass, rather
the fixed number of tokens
- set a percentage : 0.9
- model sorts the tokens by probability and keeps adding them to a pool until the cumulaitve
sum of their probabilities hist 90%
    - if the model is highly confident: it will take less tokens to reach 90%
    - if the model is uncertain: it will take more tokens to reach 90%
- context aware filtering
- superior, produces natural sounding, diverse text
- top_p only effects sampling , it doesnot effect greedy decoding much
- for top_p to take effect you must use temperature>0

In [61]:

def generate(model, idx, attention_mask,max_new_tokens, context_size,top_p=None,top_k=None,temperature=0.0, eos_id=None):
    batch_size=idx.shape[0]
    finished=torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=idx.device
    )
    for _ in range(max_new_tokens):
        active_mask=~finished
        active_indices=torch.nonzero(active_mask,as_tuple=False).squeeze(-1)
        if active_indices.numel()==0:
            break
        active_idx=idx[active_indices]
        active_attention_mask=attention_mask[active_indices]
        idx_cond=active_idx[:,-context_size:]
        attn_cond=active_attention_mask[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond,attention_mask=attn_cond)
        # only select the last token - newly generated token
        logits=logits[:,-1,:]
        if top_k is not None:
            # only cares about ranking not probs
            top_logits,_=torch.topk(logits,top_k)
            min_val=top_logits[:,-1].unsqueeze(-1)
            logits=torch.where(logits<min_val,torch.tensor(float("-inf")).to(logits.device),logits)
        if top_p is not None:
            # renormalize the remaining tokens probs 
            probs=torch.softmax(logits,dim=-1)
            sorted_probs,sorted_indices=torch.sort(
                probs,
                descending=True,
                dim=-1
            )
            cumulative_probs=torch.cumsum(sorted_probs,dim=-1)
            sorted_indices_to_remove=cumulative_probs>top_p
            # shift by one - include the token that actually crossed
            # the threshold
            sorted_indices_to_remove[..., 1:]=sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = False
            indices_to_remove=torch.zeros_like(logits,dtype=torch.bool)
            indices_to_remove.scatter_(
                dim=-1,
                index=sorted_indices,
                src=sorted_indices_to_remove
            )
            logits = logits.masked_fill(
                indices_to_remove,
                float("-inf")
            )
        if temperature>0.0:
            logits=logits/temperature
            logits=logits-logits.max(dim=-1,keepdim=True).values
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
        else:
            # it temp is 0. its same as greedy 
            idx_next=torch.argmax(logits,dim=-1,keepdim=True)
        new_tokens=torch.full(
            (batch_size,1),
            eos_id if eos_id is not None else 0,
            dtype=idx.dtype,
            device=idx.device
        )
        new_tokens[active_indices]=idx_next
        idx=torch.cat((idx,new_tokens),dim=1)

        new_attention=torch.zeros(
            (batch_size,1),
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        new_attention[active_indices]=1
        attention_mask=torch.cat((attention_mask,new_attention),dim=1)
        if eos_id is not None:
            newly_finished = (
                idx_next.squeeze(-1) == eos_id
            )
            finished[active_indices]|=newly_finished
        if finished.all():
            break
    return idx

In [68]:
input_ids, attention_mask = texts_to_token_ids(
    [
        "Every effort moves",
        "practice makes man"
    ],
    tokenizer
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

token_ids = generate(
    model=gpt,
    idx=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    # top_k=40,
    top_p=0.8,
    temperature=0.3,
    eos_id=tokenizer.eot_token
)

In [69]:
print("generated output: \n")
for i,out in enumerate(token_ids_to_texts(token_ids,tokenizer)):
    print(f"""{i+1}. \n{out}\n""")

generated output: 

1. 
Every effort moves forward, but the next step is to try to get the best of the best.

"We have to make sure that we're not going

2. 
practice makes man's life more difficult, but it's also a good thing.

The best way to learn to be a better man is to be a better



## min-p sampling

- you set a min_p
- the model looks at the prob of the absolute top token
- cutoff for all tokens = min_p multiplied by the top token's prob
- balances strictness and creativity. aggressively prunes bad options when the model is confident

    logits
    ↓
    temperature scaling
    ↓
    top_k
    ↓
    top_p
    ↓
    min_p
    ↓
    softmax
    ↓
    multinomial

    LOGIT SPACE
    ↓
    PROBABILITY SPACE
    ↓
    SAMPLING

## logits space
    Temperature
    Top-k
    Repetition penalty
    Presence penalty
    Frequency penalty
    Logit bias
    Bad-word masking
    Forced tokens

## probs space
    Top-p (nucleus)
    Min-p
    Typical sampling
    Entropy-based methods

## modern decoding methods
    Create a mask
    ↓
    Set masked logits to -inf
    ↓
    Final softmax
    ↓
    Sample

In [70]:

def generate(model, idx, attention_mask,max_new_tokens, context_size,min_p=None,top_p=None,top_k=None,temperature=0.0, eos_id=None):
    batch_size=idx.shape[0]
    finished=torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=idx.device
    )
    for _ in range(max_new_tokens):
        active_mask=~finished
        active_indices=torch.nonzero(active_mask,as_tuple=False).squeeze(-1)
        if active_indices.numel()==0:
            break
        active_idx=idx[active_indices]
        active_attention_mask=attention_mask[active_indices]
        idx_cond=active_idx[:,-context_size:]
        attn_cond=active_attention_mask[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond,attention_mask=attn_cond)
        # only select the last token - newly generated token
        # batchsize, 1, logits
        logits=logits[:,-1,:]
        if temperature>0.0:
            logits=logits/temperature
        if top_k is not None:
            top_logits,_=torch.topk(logits,top_k)
            min_val=top_logits[:,-1].unsqueeze(-1)
            logits=torch.where(logits<min_val,torch.tensor(float("-inf")).to(logits.device),logits)
        if top_p is not None:
            # renormalize the remaining tokens probs 
            probs=torch.softmax(logits,dim=-1)
            sorted_probs,sorted_indices=torch.sort(
                probs,
                descending=True,
                dim=-1
            )
            cumulative_probs=torch.cumsum(sorted_probs,dim=-1)
            sorted_indices_to_remove=cumulative_probs>top_p
            # shift by one - include the token that actually crossed
            # the threshold
            sorted_indices_to_remove[..., 1:]=sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = False
            indices_to_remove=torch.zeros_like(logits,dtype=torch.bool)
            indices_to_remove.scatter_(
                dim=-1,
                index=sorted_indices,
                src=sorted_indices_to_remove
            )
            logits = logits.masked_fill(
                indices_to_remove,
                float("-inf")
            )
        if min_p is not None:
            # (batchsize,1,1)
            probs=torch.softmax(logits,dim=-1)
            max_prob_values,_=torch.max(probs, dim=-1,keepdim=True)
            threshold=max_prob_values*min_p
            indices_to_remove= probs < threshold
            logits=logits.masked_fill(
                indices_to_remove,
                float("-inf")
            )
        # token selection
        if temperature>0.0:
            logits=logits-logits.max(dim=-1,keepdim=True).values
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
        else:
            # it temp is 0. its same as greedy 
            idx_next=torch.argmax(logits,dim=-1,keepdim=True)
        
        new_tokens=torch.full(
            (batch_size,1),
            eos_id if eos_id is not None else 0,
            dtype=idx.dtype,
            device=idx.device
        )
        new_tokens[active_indices]=idx_next
        idx=torch.cat((idx,new_tokens),dim=1)

        new_attention=torch.zeros(
            (batch_size,1),
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        new_attention[active_indices]=1
        attention_mask=torch.cat((attention_mask,new_attention),dim=1)
        if eos_id is not None:
            newly_finished = (
                idx_next.squeeze(-1) == eos_id
            )
            finished[active_indices]|=newly_finished
        if finished.all():
            break
    return idx

In [71]:
input_ids, attention_mask = texts_to_token_ids(
    [
        "Every effort moves",
        "practice makes man"
    ],
    tokenizer
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

token_ids = generate(
    model=gpt,
    idx=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    min_p=0.1,
    top_p=0.9,
    temperature=0.7,
    eos_id=tokenizer.eot_token
)

In [72]:
print("generated output: \n")
for i,out in enumerate(token_ids_to_texts(token_ids,tokenizer)):
    print(f"""{i+1}. \n{out}\n""")

generated output: 

1. 
Every effort moves forward, and we will continue to build on our strong partnership with the United States," the statement said. "We will continue to work closely with our

2. 
practice makes man's life easier, and it's the best way to get rid of the stress.

The best way to get rid of stress is to get



## beam search (search algorithm not a sampling algorithm)
- deterministic search and decoding algorithm
-  A deterministic (non-random) search algorithm. Instead of picking just one word at a time, it keeps track of a set number of the most promising "paths" (word sequences) simultaneously at every step.
-  calculates the most probable sequences of tokens over multiple steps
-  `deterministc, not a random sampler`
-  set a beam width: b
    -  instead of just taking the absolute best token
    -  the model keeps track of the top 3 tokens alive as independent "branches"
    -  or "beams"
    -  now model calculates the next token probs for all the three branches
    -  calculates the cumulative log-probability for everypath
    -  sorts all new paths and keep the top 3 with the hightest cumulative scores and discards the rest
- exhaustive and calculated
- reduces the chance of a model painting itself into a corner, gold standard for tasks where there is a definitive correct answer - machine translation or text summarization
- computationally expensive
- open-ended generation - produces boring, repititve and generic text
- Why It Works Well: when there is a relatively correct answer:
    - Machine Translation
    - Speech Recognition
    - OCR
    - Summarization
    - Code completion in constrained settings
- by default :doesnot create diversity,it creates multiple high-possibility variant
- multiple high-probability variants

    Current beams
    ↓
    Expand each beam
    ↓
    beam_size² candidates
    ↓
    Sort by cumulative score
    ↓
    Keep best beam_size
    ↓
    Repeat

In [79]:
torch.empty(3,1)

tensor([[-8.9234e-10],
        [ 3.0946e-41],
        [ 0.0000e+00]])

In [118]:
torch.zeros(1,).shape

torch.Size([1])

In [20]:
eos_token_id=tokenizer.eot_token

### EOS Masking Trick: 
When a sequence hits <eos>, we don't want it to keep generating random tokens, but we also can't just delete it from the tensor (because tensor shapes must remain uniform). By setting its future probabilities to -inf and the EOS token probability to 0.0, the sequence "idles", generating <eos> over and over without worsening its cumulative score.

### Length Normalization: 
Cumulative log-probabilities are strictly negative. Therefore, a 20-token sequence will naturally have a lower (more negative) score than a 5-token sequence, even if it's a better translation/generation. Dividing by $L^\alpha$ offsets this bias.

### Beam History: 
Appending the cloned state at each step allows you to map out the tree structure later.

In [16]:
@torch.no_grad()
def beam_search_generate(model,idx,attention_mask,beam_size,max_new_tokens,context_size,length_penalty=1.0, return_num_beams=1):
    # lets solve for only batch size of 1(1, seq_len)
    # at the start: 1, seq_len
    # at the start, one beam alive, score=0
    beams=idx
    scores=torch.zeros(1,device=idx.device)
    finished=torch.zeros(
        beams.shape[0],
        dtype=torch.bool,
        device=beams.device
    )
    beam_history=[]
    for step in range(max_new_tokens):
        beam_cond = beams[:, -context_size:]

        mask_cond = attention_mask[:, -context_size:]
        # num_beams, seq_len, vocab_size
        logits=model(beam_cond,attention_mask=mask_cond)
        # num_beams,vocab_size
        logits=logits[:,-1,:]
        # num_beams,vocab_size
        log_probs=torch.log_softmax(logits,dim=-1)

        if finished.any():
            finished_mask=finished.unsqueeze(-1).expand_as(log_probs)
            log_probs=log_probs.masked_fill(finished_mask,-float("inf"))
            log_probs[finished,eos_token_id]=0.0
        # num_beams, beam_size
        topk_log_probs,topk_tokens=torch.topk(log_probs,beam_size,dim=-1)
        # update beam scores -> num_beams, beam_size
        # cumulative scores
        candidate_scores=scores.unsqueeze(1)+topk_log_probs
        # (num_beams, beam_size, seq_len)
        # expand beams and masks
        expaned_beams=beams.unsqueeze(1).repeat(1,beam_size,1)
        expanded_masks=attention_mask.unsqueeze(1).repeat(1,beam_size,1)
        # num_beams, beam_size,1
        new_tokens= topk_tokens.unsqueeze(-1)
        new_mask_tokens=torch.ones(new_tokens.shape[0],beam_size,1,device=attention_mask.device,dtype=attention_mask.dtype)
        
        # (num_beams, beam_size, seq_len+1)
        candidate_beams=torch.cat([expaned_beams,new_tokens],dim=-1)
        candidate_masks=torch.cat([expanded_masks,new_mask_tokens],dim=-1)
        # flatten for selection
        # (num_beams*beam_size,seq_len+1)
        candidate_beams=candidate_beams.view(-1,candidate_beams.shape[-1])
        candidate_masks=candidate_masks.view(-1,candidate_masks.shape[-1])
        # (num_beams*beam_size)
        candidate_scores=candidate_scores.view(-1)
        # select top-k across all expanded beams
        # beam_size
        top_scores,top_indices=torch.topk(candidate_scores,beam_size)
        beams=candidate_beams[top_indices]
        attention_mask=candidate_masks[top_indices]
        scores=top_scores
        # update finished state
        finished=(beams[:,-1]==eos_token_id)
        beam_history.append(
            {
                "step":step,
                "beams":beams.clone(),
                "scores":scores.clone()
            }
            
        )
        if finished.all():
            break
    # length normalization
    seq_lengths =(beams!=eos_token_id).sum(dim=-1).float()

    # formula: score / (length ^ penalty). 
    # penalty > 0 favors longer sequences (since scores are negative).

    normalized_scores=scores/ (seq_lengths**length_penalty)
    # topn beams + scores
    sorted_scores,sorted_indices=torch.sort(normalized_scores,descending=True)
    best_beams=beams[sorted_indices][:return_num_beams]
    best_scores=sorted_scores[:return_num_beams]
    return best_beams, best_scores,beam_history

#### blog example

In [162]:
print("generated output: \n")
for i,out in enumerate(token_ids_to_texts(token_ids,tokenizer)):
    print(f"""{i+1}. \n{out}\n""")

generated output: 

1. 
Every effort moves forward.

"I think we're going to be able to do a lot of things that we're not able to do right now," he

2. 
Every effort moves forward.

"I think we're going to be able to do a lot of things that we're not able to do in the past,"

3. 
Every effort moves forward.

"I think we're going to be able to do a lot of things that we're not able to do right now," said

4. 
Every effort moves forward.

"I think we're going to be able to do a lot of things that we're not able to do right now. We



1. Why decoding matters
2. Greedy decoding
3. Temperature sampling
4. Top-K sampling
5. Top-P sampling
6. Min-P sampling
7. Beam search
8. Comparing outputs
9. When to use what

In [18]:
input_ids, attention_mask = texts_to_token_ids(
    [
        "Every effort moves",
        # "practice makes man"
    ],
    tokenizer
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

In [35]:
# Using the new beam search generator
best_beams, best_scores, beam_history = beam_search_generate(
    model=gpt,
    idx=input_ids,
    attention_mask=attention_mask,
    beam_size=3,                           # Tracks top 3 paths per batch item
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    length_penalty=1.2,                    # > 1.0 encourages longer sequences before hitting EOS
    return_num_beams=3                     # Returns the single best path per prompt
)

In [36]:
# Decoding the outputs
for i in range(input_ids.shape[0]):
    # Assuming the implementation returns shape (batch_size, return_num_beams, seq_len) 
    # or (batch_size, seq_len) if return_num_beams=1
    
    # If the function collapses to 2D for return_num_beams=1:
    generated_sequence = best_beams[i] 
    
    # If the function keeps the 3D shape (batch_size, return_num_beams, seq_len):
    # generated_sequence = best_beams[i][0] 
    
    text = tokenizer.decode(generated_sequence.tolist())
    score = best_scores[i].item()
    print(f"Batch {i+1} (Score: {score:.3f}):\n{text}\n")

Batch 1 (Score: -0.514):
Every effort moves forward.

"I'm not going to say it's going to be easy," he said. "I'm not going to say it's

